In [1]:
# STEP 1: Load data
# -----------------
import pandas as pd
import json
import ast

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_ach_natural.csv"
matrix_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/Construction_Database.csv"

# Load datasets
epc_df = pd.read_csv(epc_path)
matrix_df = pd.read_csv(matrix_path)

print("EPC shape:", epc_df.shape)
print("Matrix shape:", matrix_df.shape)

# Preview
display(epc_df.head())
display(matrix_df.head())

EPC shape: (117, 229)
Matrix shape: (55, 14)


,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS,WINDOW_AREA_SINGLE,ASPECT_RATIO,WINDOW_H,WINDOW_W,EXISTING_PV_PANELS,MODELED_PV,PV_UPGRADE,ACH_NATURAL
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,100,170,...,2,0,1.790829,1.75,1.011598,1.770297,0,0,1,1.05
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,180,90,...,3,8,3.189657,2.25,1.190641,2.678942,0,8,1,1.01
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,100,170,...,3,2,2.044886,1.75,1.080975,1.891706,0,2,1,0.82
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,150,120,...,3,8,3.295329,2.25,1.210203,2.722956,0,8,1,1.22
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,150,120,...,5,21,2.763486,2.25,1.108249,2.493560,0,21,1,0.84


,Description,A,B,C,D,E,F,G,H,I,J,K,L,M
0,Average thermal transmittance 0.12 W/m2K,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12,avg_u0.12
1,Average thermal transmittance 0.13 W/m2K,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13,avg_u0.13
2,Average thermal transmittance 0.21 W/m2K,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21,avg_u0.21
3,"Cavity wall, as built, insulated (assumed)",0,0,0,0,cavity_as_built_bands_A_to_E,0,cavity_as_built_band_G,0,0,0,0,0,0
4,"Cavity wall, as built, no insulation (assumed)",cavity_as_built_bands_A_to_E,0,cavity_as_built_bands_A_to_E,cavity_as_built_bands_A_to_E,cavity_as_built_bands_A_to_E,0,0,0,0,0,0,0,0


In [2]:
# STEP 2: Build lookup dictionary
# ------------------------------

# Ensure first column is named consistently
matrix_df = matrix_df.rename(columns={matrix_df.columns[0]: "DESCRIPTION"})

# Melt into long format
matrix_long = matrix_df.melt(
    id_vars="DESCRIPTION",
    var_name="AGE_BAND",
    value_name="CONSTRUCTION"
)

# Clean values
matrix_long["CONSTRUCTION"] = matrix_long["CONSTRUCTION"].astype(str).str.strip()
matrix_long["DESCRIPTION"] = matrix_long["DESCRIPTION"].str.strip()

# Remove zeros (treated as no mapping)
matrix_long = matrix_long[matrix_long["CONSTRUCTION"] != "0"]

# Create lookup dict
lookup = {
    (row["DESCRIPTION"], row["AGE_BAND"]): row["CONSTRUCTION"]
    for _, row in matrix_long.iterrows()
}

print("Total mappings:", len(lookup))

Total mappings: 468


In [3]:
# STEP 3: Parse JSON columns
# --------------------------

def extract_description(cell):
    if pd.isna(cell):
        return None
    
    try:
        # Try JSON first
        data = json.loads(cell)
    except:
        try:
            # Fallback to Python literal
            data = ast.literal_eval(cell)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        desc = data[0].get("description", None)
        
        if desc:
            # Clean "Description: ..." prefix
            desc = desc.replace("Description:", "").strip()
        
        return desc
    
    return None

# Apply extraction
epc_df["WALL_DESC"] = epc_df["WALL_DESCRIPTION_PARSED"].apply(extract_description)
epc_df["ROOF_DESC"] = epc_df["ROOF_DESCRIPTION_PARSED"].apply(extract_description)
epc_df["WINDOW_DESC"] = epc_df["WINDOWS_DESCRIPTION_PARSED"].apply(extract_description)

display(epc_df[["WALL_DESC", "ROOF_DESC", "WINDOW_DESC"]].head())

,WALL_DESC,ROOF_DESC,WINDOW_DESC
0,"Cavity wall, filled cavity","Pitched, 300 mm loft insulation",Fully double glazed
1,"Timber frame, as built, insulated (assumed)","Pitched, 300 mm loft insulation",Fully double glazed
2,"Cavity wall, filled cavity","Pitched, 200 mm loft insulation",Fully double glazed
3,"Cavity wall, as built, no insulation (assumed)","Pitched, 250 mm loft insulation",Fully double glazed
4,"Cavity wall, as built, insulated (assumed)","Pitched, 200 mm loft insulation",Fully double glazed


In [4]:
# STEP 4: Assignment function
# ---------------------------

def assign_construction(description, age_band):
    
    # Case 1: Missing data
    if pd.isna(description) or pd.isna(age_band):
        return "ND"
    
    key = (description.strip(), str(age_band).strip())
    
    # Case 2: Found mapping
    if key in lookup:
        return lookup[key]
    
    # Case 3: Not in matrix
    return "NIM"

In [5]:
# STEP 5: Apply mapping
# ---------------------

epc_df["WALL_CONS"] = epc_df.apply(
    lambda row: assign_construction(row["WALL_DESC"], row["AGE_BAND"]),
    axis=1
)

epc_df["ROOF_CONS"] = epc_df.apply(
    lambda row: assign_construction(row["ROOF_DESC"], row["AGE_BAND"]),
    axis=1
)

epc_df["WINDOW_CONS"] = epc_df.apply(
    lambda row: assign_construction(row["WINDOW_DESC"], row["AGE_BAND"]),
    axis=1
)

display(epc_df[[
    "WALL_DESC", "WALL_CONS",
    "ROOF_DESC", "ROOF_CONS",
    "WINDOW_DESC", "WINDOW_CONS"
]].head())

,WALL_DESC,WALL_CONS,ROOF_DESC,ROOF_CONS,WINDOW_DESC,WINDOW_CONS
0,"Cavity wall, filled cavity",cavity_filled_bands_A_to_E,"Pitched, 300 mm loft insulation",insulated_roof_300mm,Fully double glazed,dbl_glz_pre_2003
1,"Timber frame, as built, insulated (assumed)",timber_as_built_bands_G_to_I,"Pitched, 300 mm loft insulation",insulated_roof_300mm,Fully double glazed,dbl_glz_pre_2003
2,"Cavity wall, filled cavity",cavity_filled_bands_A_to_E,"Pitched, 200 mm loft insulation",insulated_roof_200mm,Fully double glazed,dbl_glz_pre_2003
3,"Cavity wall, as built, no insulation (assumed)",cavity_as_built_bands_A_to_E,"Pitched, 250 mm loft insulation",insulated_roof_250mm,Fully double glazed,dbl_glz_pre_2003
4,"Cavity wall, as built, insulated (assumed)",cavity_as_built_band_G,"Pitched, 200 mm loft insulation",insulated_roof_200mm,Fully double glazed,dbl_glz_pre_2003


In [6]:
# STEP 6: Diagnostics
# -------------------

print("WALL NIM:", (epc_df["WALL_CONS"] == "NIM").sum())
print("ROOF NIM:", (epc_df["ROOF_CONS"] == "NIM").sum())
print("WINDOW NIM:", (epc_df["WINDOW_CONS"] == "NIM").sum())

print("WALL ND:", (epc_df["WALL_CONS"] == "ND").sum())
print("ROOF ND:", (epc_df["ROOF_CONS"] == "ND").sum())
print("WINDOW ND:", (epc_df["WINDOW_CONS"] == "ND").sum())

# Inspect mismatches
display(epc_df[epc_df["WALL_CONS"] == "NIM"][["WALL_DESC", "AGE_BAND"]].head(10))

WALL NIM: 1
ROOF NIM: 0
WINDOW NIM: 0
WALL ND: 0
ROOF ND: 0
WINDOW ND: 0


,WALL_DESC,AGE_BAND
30,"System built, with external insulation",C


In [7]:
# STEP 7: Save updated dataset
# ----------------------------

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_cons_update6.csv"

epc_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_cons_update6.csv
